In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
import copy
import importlib


In [ ]:
# Import Required Libraries
import numpy as np
import random
import matplotlib.pyplot as plt
import time

# Constants and Hyperparameters
N_ROWS, N_COLS = 25, 25
START = (0, 0)
GOAL = (N_ROWS  - 1, N_COLS - 1)

# Cliff area - bottom rows between start and goal (like classic CliffWalk)
# CLIFF = [
#     (N_ROWS - 1, col) for col in range(1, N_COLS - 1)
# ] + [
#     (N_ROWS - 6, col) for col in range(1, N_COLS - 1)
# ] + [
#     (N_ROWS - 10, col) for col in range(1, N_COLS - 1)
# ] 
CLIFF = []
WALLS = [
    
   (N_ROWS - 1, 21), 
   (N_ROWS - 2, 22),
   (N_ROWS - 3, 22),
   (N_ROWS - 4, 22),
   (N_ROWS - 5, 22),
   (N_ROWS - 6, 22),
   (N_ROWS - 7, 22),
   (N_ROWS - 8, 22),
   (N_ROWS - 9, 22),
   (N_ROWS - 10, 22),
   (N_ROWS - 11, 22),
   (N_ROWS - 12, 22),
   (N_ROWS - 13, 21), 
   (N_ROWS - 14, 22),
   (N_ROWS - 15, 22),
   (N_ROWS - 16, 22),
   (N_ROWS - 17, 22),
   (N_ROWS - 18, 22),
   (N_ROWS - 19, 22),
   
#    (N_ROWS - 10, 21),
#    (N_ROWS - 10, 20), 
#    (N_ROWS - 10, 19), 
#    (N_ROWS - 10, 18),
#    (N_ROWS - 10, 17),
#    (N_ROWS - 10, 16), 
#    (N_ROWS - 10, 15),
#    (N_ROWS - 10, 14), 
#    (N_ROWS - 10, 10),

#     # Additional obstacles - preserved path to goal (no blocks on col 23+)
#     # Upper-left vertical segment (col 4, rows 2-8)
#     (2, 4),
#     (3, 4),
#     (4, 4),
#     (5, 4),
#     (6, 4),
#     (7, 4),
#     (8, 4),

#     # Upper band horizontal segment (row 8, cols 6-12)
#     (8, 6),
#     (8, 7),
#     (8, 8),
#     (8, 9),
#     (8, 10),
#     (8, 11),
#     (8, 12),

#     # Central 3x3 block (rows 11-13, cols 6-8)
#     (11, 6),
#     (11, 7),
#     (11, 8),
#     (12, 6),
#     (12, 7),
#     (12, 8),
#     (13, 6),
#     (13, 7),
#     (13, 8),

#     # Right-upper vertical segment (col 18, rows 2-5)
#     (2, 18),
#     (3, 18),
#     (4, 18),
#     (5, 18),
#    (N_ROWS - 10, 12), 
#    (N_ROWS - 10, 11), 
#    (N_ROWS - 10, 10)
]
# actions
# Action definitions
ACTIONS_4 = { # row = y, col = x
    0: (-1, 0),  # Up
    1: ( 1, 0),  # Down
    2: ( 0,-1),  # Left
    3: ( 0, 1),  # Right
}

ACTIONS_5 = dict(ACTIONS_4)
ACTIONS_5[4] = (1, 1)  # south-east (down-right)

ACTIONS_8 = dict(ACTIONS_5)
ACTIONS_8[5] = (1, -1)  # down-left (south-west)
ACTIONS_8[6] = (-1, 1)  # top-right (north-east)
ACTIONS_8[7] = (-1, -1)  # top-left (north-west)

class GridWorld:
    def __init__(self, n_rows, n_cols, start, goal, walls, step_reward, goal_reward, bump_reward, gamma, cliff = CLIFF, cliff_reward=-200):
        self.n_rows = n_rows
        self.n_cols = n_cols
        self.start = start
        self.goal = goal
        self.walls = set(walls)
        self.cliff = set(cliff) if cliff is not None else set()
        self.step_reward = step_reward
        self.goal_reward = goal_reward
        self.gamma = gamma
        self.bump_reward = bump_reward
        self.cliff_reward = cliff_reward
        self.success_prob = 0.7
        self.noise_prob = 0.1
        self.stay_prob = 0.2
        
    def in_bounds(self, state):
        r, c = state
        return 0 <= r < self.n_rows and 0 <= c < self.n_cols
    
    def step2(self, state, action, actions, rng=None):
        if state == self.goal:
            return state, 0.0, True
            
        if rng is None:
            rng = np.random.default_rng()
        
        # Determine actual action based on transition probabilities
        rand_val = rng.random()
        
        if rand_val < self.success_prob:
            # Intended action
            actual_action = action
        elif rand_val < self.success_prob + self.noise_prob:
            # Random action from available actions
            actual_action = rng.integers(0, len(actions))
        else:
            # Stay in place (no action)
            actual_action = None
        
        if actual_action is None:
            # Stay in current state
            next_state = state
        else:
            # Execute the actual action
            dr, dc = actions[actual_action]
            next_state = (state[0] + dr, state[1] + dc)
            if not self.in_bounds(next_state) or next_state in self.walls:
                next_state = state  # Stay put if invalid
        
        # Check if agent fell off cliff
        if next_state in self.cliff:
            return self.start, self.cliff_reward, False
        
        reward = self.goal_reward if next_state == self.goal else self.step_reward
        done = next_state == self.goal
        return next_state, reward, done
    
    def step(self, state, action, actions, rng=None):
        if state == self.goal:
            return state, 0.0, True
        
        # Deterministic action execution - always execute the intended action
        dr, dc = actions[action]
        next_state = (state[0] + dr, state[1] + dc)
        
        # If next state is out of bounds or hits a wall, stay in current state
        if not self.in_bounds(next_state) or next_state in self.walls:
            next_state = state
        
        # Check if agent fell off cliff
        if next_state in self.cliff:
            return state, self.cliff_reward, False
        
        if next_state == state:
            reward = self.bump_reward
        else:
            reward = self.goal_reward if next_state == self.goal else self.step_reward
        done = next_state == self.goal
        return next_state, reward, done

    def to_index(self, state):
        r, c = state
        return r * self.n_cols + c

    def from_index(self, index):
        r = index // self.n_cols
        c = index % self.n_cols
        return (r, c)

In [ ]:

class QLearningAgent:
    def __init__(self, grid_world, n_actions, episodes=600, alpha=0.5, 
                 eps_start=1.0, eps_end=0.05, eps_decay_episodes=300, 
                 max_steps=200, seed=123):
        self.grid_world = grid_world
        self.n_actions = n_actions
        self.episodes = episodes
        self.alpha = alpha
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay_episodes = eps_decay_episodes
        self.max_steps = max_steps
        self.seed = seed
        
        # Initialize random number generators
        self.rng = np.random.default_rng(seed)
        random.seed(seed)
        np.random.seed(seed)
        
        # Initialize Q-table and tracking arrays
        self.num_states = grid_world.n_rows * grid_world.n_cols
        self.Q = np.zeros((self.num_states, n_actions), dtype=float)
        self.returns = np.zeros(episodes, dtype=float)
        self.bumps = np.zeros(episodes, dtype=float)
        self.steps_arr = np.zeros(episodes, dtype=int)
        
        # Epsilon decay calculation
        self.eps_decay = (eps_start - eps_end) / max(1, eps_decay_episodes)
    
    def train(self, actions_dict, epsilon_greedy_func):
        """Train the Q-learning agent"""
        eps = self.eps_start
        
        for ep in range(self.episodes):
            s = self.grid_world.start
            si = self.grid_world.to_index(s)
            done = False
            G = 0.0
            bumpcount = 0
            disc = 1.0
            steps = 0

            for t in range(self.max_steps):
                a = epsilon_greedy_func(self.Q[si], eps, self.rng)
                s_next, r, done = self.grid_world.step(s, a, actions_dict, self.rng)
                s_next_i = self.grid_world.to_index(s_next)

                if si == s_next_i:
                    bumpcount += 1
                target = r if done else r + self.grid_world.gamma * np.max(self.Q[s_next_i])
                self.Q[si, a] += self.alpha * (target - self.Q[si, a])

                G += r
                disc *= self.grid_world.gamma
                s, si = s_next, s_next_i
                steps += 1
                if done:
                    break
            self.returns[ep] = G
            self.bumps[ep] = bumpcount
            self.steps_arr[ep] = steps
            
            if ep < self.eps_decay_episodes:
                eps = max(self.eps_end, eps - self.eps_decay)
        
        print(f'Training ({self.n_actions} actions) complete!')
    
    def get_policy(self, n_rows, n_cols):
        """Get the derived policy from Q-table"""
        return Visualizer.derive_policy(self.Q, n_rows, n_cols)
    
    def get_results(self, moving_average_func, ma_window=25):
        """Get training results with moving averages"""
        return {
            'returns': self.returns.copy(),
            'Q': self.Q.copy(),
            'bumps': self.bumps.copy(),
            'returns_ma': moving_average_func(self.returns, w=ma_window),
            'steps_ma': moving_average_func(self.steps_arr.astype(float), w=ma_window)
        }


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import torch.nn.functional as F
import numpy as np

class TransitionModelLearner(nn.Module):
    """MLP model to predict next state for diagonal actions (4-7)"""
    def __init__(self, state_dim=2, action_dim=1, hidden_dim=64, lr=0.001, buffer_size=10000):
        super(TransitionModelLearner, self).__init__()
        
        # Neural network layers
        input_dim = state_dim + action_dim  # state + action
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, state_dim)  # output_dim = state_dim
        self.dropout = nn.Dropout(0.1)
        
        # Training components
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.to(self.device)
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.criterion = nn.MSELoss()
        
        self.buffer = deque(maxlen=buffer_size) #auto remove old samples if over size
        self.min_buffer_size = 100  # Minimum samples before training
        
    def forward(self, x):
        """Forward pass through the neural network"""
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x
        
    def add_experience(self, state, action, next_state):
        # Convert grid coordinates to normalized features
        state_features = np.array([state[0] / 24.0, state[1] / 24.0])  # Normalize to [0,1]
        # Normalize action (4-7 -> 0-3)
        action_features = np.array([(action - 4) / 3.0])  # Normalize diagonal actions to [0,1]
        next_state_features = np.array([next_state[0] / 24.0, next_state[1] / 24.0])
        self.buffer.append((state_features, action_features, next_state_features))
    
    def can_predict(self):
        return len(self.buffer) >= self.min_buffer_size
    
    def train_model(self, batch_size=32, epochs=10):
        if len(self.buffer) < self.min_buffer_size:
            return
        states = []
        actions = []
        next_states = []
        sample_size = min(len(self.buffer), 1000)  #  last 1000 samples
        samples = list(self.buffer)[-sample_size:]
        
        for state_feat, action_feat, next_state_feat in samples:
            states.append(state_feat)
            actions.append(action_feat)
            next_states.append(next_state_feat)
        
        states = torch.FloatTensor(np.array(states)).to(self.device)
        actions = torch.FloatTensor(np.array(actions)).to(self.device)
        next_states = torch.FloatTensor(np.array(next_states)).to(self.device)
        self.train()
        for _ in range(epochs):
            indices = torch.randperm(len(states))
            for i in range(0, len(states), batch_size):
                batch_indices = indices[i:i+batch_size]
                batch_states = states[batch_indices]
                batch_actions = actions[batch_indices]
                batch_next_states = next_states[batch_indices]
                # Concatenate state and action
                batch_input = torch.cat([batch_states, batch_actions], dim=1)
                # forward 
                predicted_next_states = self(batch_input)
                loss = self.criterion(predicted_next_states, batch_next_states)
                
                # backward 
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
    
    def predict_next_state(self, state, action):
        self.eval()
        with torch.no_grad():
            state_features = np.array([state[0] / 24.0, state[1] / 24.0])
            action_features = np.array([(action - 4) / 3.0])  # Normalize diagonal action
            input_features = np.concatenate([state_features, action_features])
            input_tensor = torch.FloatTensor(input_features).unsqueeze(0).to(self.device)
            
            predicted = self(input_tensor)
            predicted_np = predicted.cpu().numpy()[0]
            
            # convert back to grid coordinates
            next_r = int(np.clip(predicted_np[0] * 24, 0, 24))
            next_c = int(np.clip(predicted_np[1] * 24, 0, 24))
            
            return (next_r, next_c)

In [ ]:
import numpy as np

class MLPQLearningAgent(QLearningAgent):
    """Q-learning agent with learned MLP model for action adaptation"""
    
    def __init__(self, grid_world, n_actions, base_q_table=None, episodes=600, alpha=0.5, 
                 eps_start=1.0, eps_end=0.05, eps_decay_episodes=300, 
                 max_steps=200, seed=123, use_model=True, use_conditional=True, use_oracle=False):
        super().__init__(grid_world, n_actions, episodes, alpha, eps_start, eps_end, 
                        eps_decay_episodes, max_steps, seed)
        self.reuse_count = np.zeros(self.episodes, dtype=float)
        self.use_model = use_model
        self.use_conditional = use_conditional
        self.use_oracle = use_oracle
        self.base_q_table = base_q_table
        # Initialize transition model learner
        self.transition_learner = TransitionModelLearner()
        
        # Initialize with base Q-table if provided
        if base_q_table is not None:
            self.Q[:, :base_q_table.shape[1]] = base_q_table
            # Optimistic initialization for new actions
            if use_model:
                V_old = np.min(base_q_table, axis=1) 
                for new_action in range(base_q_table.shape[1], n_actions):
                    self.Q[:, new_action] = V_old
    
    def enhanced_epsilon_greedy(self, q_row, epsilon, rng, encourage_new_action=False):
        if encourage_new_action and len(self.transition_learner.buffer) < 200:
            if rng.random() < 0.3:  
                # Randomly choose one of the diagonal actions (4-7)
                return int(rng.integers(4, len(q_row)))
        
        # Standard epsilon-greedy
        if rng.random() < epsilon:
            return int(rng.integers(len(q_row)))

        max_q = np.max(q_row)
        best = np.flatnonzero(q_row == max_q)
        return int(rng.choice(best))
    
    def train_with_learned_model(self, actions_dict, epsilon_greedy_func, oracle_func):
        eps = self.eps_start
        model_train_frequency = 20  
        
        for ep in range(self.episodes):
            s = self.grid_world.start
            si = self.grid_world.to_index(s)
            done = False
            G = 0.0
            bumpcount = 0
            disc = 1.0
            steps = 0
            reuse = 0
            
            for t in range(self.max_steps):
                a = epsilon_greedy_func(self.Q[si], eps, self.rng)
                # Use oracle or transition learner based on configuration
                if a >= 4:
                    predicted_next_state = self.transition_learner.predict_next_state(s, a)
                    predicted_next_state_i = self.grid_world.to_index(predicted_next_state)
                else:
                    predicted_next_state, _, _ = oracle_func(si, a)
                    predicted_next_state_i = self.grid_world.to_index(predicted_next_state)
                if self.use_oracle:
                    # Use oracle function to predict next state (pass state index)
                    predicted_next_state, _, _ = oracle_func(si, a)
                    predicted_next_state_i = self.grid_world.to_index(predicted_next_state)
                    if predicted_next_state_i != si and np.max(self.base_q_table[predicted_next_state_i]) > np.max(self.base_q_table[si]):
                        reuse += 1
                    else:
                        a = epsilon_greedy_func(self.Q[si], eps, self.rng)
                elif self.transition_learner.can_predict():
                    # Use learned transition model to predict next state
                    predicted_next_state = self.transition_learner.predict_next_state(s, a)
                    predicted_next_state_i = self.grid_world.to_index(predicted_next_state)
                    if predicted_next_state_i != si and np.max(self.base_q_table[predicted_next_state_i]) > np.max(self.base_q_table[si]):
                        reuse += 1
                    else:
                        a = epsilon_greedy_func(self.Q[si], eps, self.rng)
                        
                s_next, r, done = self.grid_world.step(s, a, actions_dict, self.rng)
                s_next_i = self.grid_world.to_index(s_next)

                self.transition_learner.add_experience(s.flatten(), a, s_next.flatten())

                if si == s_next_i:
                    bumpcount += 1

                target = r if done else r + self.grid_world.gamma * np.max(self.Q[s_next_i])
                self.Q[si, a] += self.alpha * (target - self.Q[si, a])

                G += r
                disc *= self.grid_world.gamma
                s, si = s_next, s_next_i
                steps += 1
                if done:
                    break
            
            self.reuse_count[ep] = reuse    
            self.returns[ep] = G
            self.bumps[ep] = bumpcount
            self.steps_arr[ep] = steps
            
            if ep > 0 and ep % model_train_frequency == 0 and len(self.transition_learner.buffer) > 50:
                self.transition_learner.train_model(batch_size=32, epochs=5)
            
            if ep < self.eps_decay_episodes:
                eps = max(self.eps_end, eps - self.eps_decay)

In [ ]:

import numpy as np

class MLPQLearningAgentUseCondition(QLearningAgent):
    """Q-learning agent with learned MLP model for action adaptation"""
    
    def __init__(self, grid_world, n_actions, base_q_table=None, episodes=600, alpha=0.5, 
                 eps_start=1.0, eps_end=0.05, eps_decay_episodes=300, 
                 max_steps=200, seed=123, use_model=True, use_conditional=True,
                 exploration_start=1.0, exploration_end=0.0, exploration_decay_episodes=300):
        super().__init__(grid_world, n_actions, episodes, alpha, eps_start, eps_end, 
                        eps_decay_episodes, max_steps, seed)
        self.reuse_count = np.zeros(self.episodes, dtype=float)
        self.reject_count = np.zeros(self.episodes, dtype = float)
        self.use_model = use_model
        self.use_conditional = use_conditional
        self.base_q_table = base_q_table
        
        # Exploration encouragement decay parameters
        self.exploration_start = exploration_start
        self.exploration_end = exploration_end
        self.exploration_decay_episodes = exploration_decay_episodes
        self.exploration_decay = (exploration_start - exploration_end) / exploration_decay_episodes if exploration_decay_episodes > 0 else 0
        # Initialize transition model learner
        self.transition_learner = TransitionModelLearner()
        
        # Initialize with base Q-table if provided
        if base_q_table is not None:
            self.Q[:, :base_q_table.shape[1]] = base_q_table
            # Optimistic initialization for new actions
            V_old = np.min(base_q_table, axis=1) 
            if use_model:
                for new_action in range(base_q_table.shape[1], n_actions):
                    self.Q[:, new_action] = V_old
    
    def enhanced_epsilon_greedy(self, q_row, epsilon, rng, encourage_new_action=False):
        if encourage_new_action:
            if rng.random() < 0.3:  
                # Randomly choose one of the diagonal actions (4-7)
                return int(rng.integers(4, len(q_row)))
        
        # Standard epsilon-greedy
        if rng.random() < epsilon:
            return int(rng.integers(len(q_row)))

        max_q = np.max(q_row)
        best = np.flatnonzero(q_row == max_q)
        return int(rng.choice(best))
    
    def train_with_learned_model(self, actions_dict, epsilon_greedy_func):
        eps = self.eps_start
        exploration_prob = self.exploration_start
        model_train_frequency = 20  
        
        for ep in range(self.episodes):
            s = self.grid_world.start
            si = self.grid_world.to_index(s)
            done = False
            G = 0.0
            bumpcount = 0
            disc = 1.0
            steps = 0
            reuse = 0
            reject = 0
            for t in range(self.max_steps):
                # Use decaying exploration probability instead of fixed threshold
                if self.rng.random() < exploration_prob:  
                    a = self.enhanced_epsilon_greedy(self.Q[si], eps, self.rng, encourage_new_action=True) # This part encourages exploration of new actions 
                else:
                    a = epsilon_greedy_func(self.Q[si], eps, self.rng, encourage_new_action = True, epissodes=ep)
                
                if a >=4 and self.transition_learner.can_predict(): 
                    snext_model = self.transition_learner.predict_next_state(s, a)
                    snext_model_i = self.grid_world.to_index(snext_model)
                    if np.max(self.Q[snext_model_i]) > np.max(self.Q[si]):  
                        reuse += 1
                    else: 
                        reject += 1
                        a = epsilon_greedy_func(self.base_q_table[si], eps, self.rng)
                
                s_next, r, done = self.grid_world.step(s, a, actions_dict, self.rng)
                s_next_i = self.grid_world.to_index(s_next)

                if a >= 4:  # Any diagonal action
                    self.transition_learner.add_experience(s, a, s_next)

                if si == s_next_i:
                    bumpcount += 1

                target = r if done else r + self.grid_world.gamma * np.max(self.Q[s_next_i])
                self.Q[si, a] += self.alpha * (target - self.Q[si, a])

                G += r
                disc *= self.grid_world.gamma
                s, si = s_next, s_next_i
                steps += 1
                if done:
                    break
            
            self.reuse_count[ep] = reuse 
            self.reject_count[ep] = reject   
            self.returns[ep] = G
            self.bumps[ep] = bumpcount
            self.steps_arr[ep] = steps
            
            if ep > 0 and ep % model_train_frequency == 0 and len(self.transition_learner.buffer) > 50:
                self.transition_learner.train_model(batch_size=32, epochs=5)
            
            # Decay both epsilon and exploration probability
            if ep < self.eps_decay_episodes:
                eps = max(self.eps_end, eps - self.eps_decay)
            
            if ep < self.exploration_decay_episodes:
                exploration_prob = max(self.exploration_end, exploration_prob - self.exploration_decay)

In [ ]:
class OracleQLearningAgent(QLearningAgent):
    """Q-learning agent with oracle model for action adaptation"""
    
    def __init__(self, grid_world, n_actions, base_q_table=None, episodes=600, alpha=0.5, 
                 eps_start=1.0, eps_end=0.05, eps_decay_episodes=300, 
                 max_steps=200, seed=123, use_model = True, use_conditional = True, walls = None):
        super().__init__(grid_world, n_actions, episodes, alpha, eps_start, eps_end, 
                        eps_decay_episodes, max_steps, seed)
        self.reuse_count = np.zeros(self.episodes, dtype = float)
        self.reject_count = np.zeros(self.episodes, dtype = float)
        self.use_model = use_model
        self.base_q_table = base_q_table
        self.use_conditional = use_conditional
        self.walls = walls
        # Initialize with base Q-table if provided
        if base_q_table is not None:
            # Optimistic initialization for new actions
            self.Q[:, :base_q_table.shape[1]] = base_q_table
            if use_model == True:
                # Randomly select a value from each state's Q-values
                rng = np.random.RandomState(seed)
                # V_old = np.array([rng.choice(base_q_table[i]) for i in range(base_q_table.shape[0])])
                V_old = np.min(base_q_table, axis=1)
                print("V_old:", V_old)
                for new_action in range(base_q_table.shape[1], n_actions):
                    self.Q[:, new_action] = V_old
    
    def epsilon_greedy(q_actions, epsilon, rng):
        """Epsilon-greedy action selection"""
        if rng.random() < epsilon:
            return int(rng.integers(len(q_actions)))
        # break ties randomly among maxima
        max_q = np.max(q_actions) # 70%, 50% new actions, best
        best = np.flatnonzero(q_actions == max_q)
        return int(rng.choice(best))
    
    def is_near_walls(self, si):
        """Check if state index si is near any wall"""
        if self.walls is None or len(self.walls) == 0:
            return False
        
        s = self.grid_world.to_state(si)
        for wall in self.walls:
            # Check if state is adjacent to wall (including diagonals)
            if abs(s[0] - wall[0]) <= 2 and abs(s[1] - wall[1]) <= 2:
                return True
        return False
    
    def train_with_oracle(self, actions_dict, epsilon_greedy_func, oracle_func):
        """Train with oracle model guidance"""
        eps = self.eps_start
        for ep in range(self.episodes):
            s = self.grid_world.start
            si = self.grid_world.to_index(s)
            done = False
            G = 0.0
            bumpcount = 0
            disc = 1.0
            steps = 0
            reuse = 0
            reject = 0
            for t in range(self.max_steps):
                # Oracle guidance for new diagonal actions (actions 4-7)
                if self.use_conditional is True:
                    a = epsilon_greedy_func(self.Q[si], eps, self.rng)
                    if a >=4: 
                        snext_model, reward_predict, done = oracle_func(si, a)
                        snext_model_i = self.grid_world.to_index(snext_model)
                        # Only accept if it leads to better state value
                        # V(-1) < V(0)
                        # -1, goal is 100
                        #next_state_value = reward_predict + self.grid_world.gamma * np.max(self.Q[snext_model_i])
                        next_state_value = np.max(self.Q[snext_model_i])
                        if reward_predict + self.grid_world.gamma * next_state_value > np.max(self.Q[si]):  
                            reuse += 1
                        else: 
                            reject += 1
                            a = epsilon_greedy_func(self.Q[si], eps, self.rng)
                else:
                    a = epsilon_greedy_func(self.Q[si], eps, self.rng)    
                s_next, r, done = self.grid_world.step(s, a, actions_dict, self.rng)
                s_next_i = self.grid_world.to_index(s_next)

                if si == s_next_i:
                    bumpcount += 1

                target = r if done else r + self.grid_world.gamma * np.max(self.Q[s_next_i])
                self.Q[si, a] += self.alpha * (target - self.Q[si, a])

                G += r
                disc *= self.grid_world.gamma
                s, si = s_next, s_next_i
                steps += 1
                if done:
                    break
            self.reuse_count[ep] = reuse    
            self.reject_count[ep] = reject
            self.returns[ep] = G
            self.bumps[ep] = bumpcount
            self.steps_arr[ep] = steps
            
            if ep < self.eps_decay_episodes:
                eps = max(self.eps_end, eps - self.eps_decay)
                

    def train_with_oracle2(self, actions_dict, epsilon_greedy_func):
        """Train the Q-learning agent"""
        eps = self.eps_start
        
        for ep in range(self.episodes):
            s = self.grid_world.start
            si = self.grid_world.to_index(s)
            done = False
            G = 0.0
            bumpcount = 0
            disc = 1.0
            steps = 0

            for t in range(self.max_steps):
                a = epsilon_greedy_func(self.Q[si], eps, self.rng)
                s_next, r, done = self.grid_world.step(s, a, actions_dict, self.rng)
                s_next_i = self.grid_world.to_index(s_next)

                if si == s_next_i:
                    bumpcount += 1
                target = r if done else r + self.grid_world.gamma * np.max(self.Q[s_next_i])
                self.Q[si, a] += self.alpha * (target - self.Q[si, a])

                G += r
                disc *= self.grid_world.gamma
                s, si = s_next, s_next_i
                steps += 1
                if done:
                    break
            self.returns[ep] = G
            self.bumps[ep] = bumpcount
            self.steps_arr[ep] = steps
            
            if ep < self.eps_decay_episodes:
                eps = max(self.eps_end, eps - self.eps_decay)
        
        print(f'Training ({self.n_actions} actions) complete!')

In [ ]:
def epsilon_greedy(q_actions, epsilon, rng, encourage_new_action=False, epissodes=None):
    if encourage_new_action and epissodes is not None and epissodes < 100:
        """Encourage exploration of new actions in early episodes"""
        if rng.random() < 0.3:  
                # Randomly choose one of the diagonal actions (4-7)
                return int(rng.integers(4, len(q_actions)))
    """Epsilon-greedy action selection"""
    if rng.random() < epsilon:
        return int(rng.integers(len(q_actions)))
    # break ties randomly among maxima
    max_q = np.max(q_actions)
    best = np.flatnonzero(q_actions == max_q)
    return int(rng.choice(best))

def moving_average(x, w=20):
    """Calculate moving average"""
    if len(x) < w:
        return x.copy()
    return np.convolve(x, np.ones(w)/w, mode='valid')

# # Legacy function wrappers for compatibility
# def to_index(s):
#     return grid_world.to_index(s)

# def from_index(i):
#     return grid_world.from_index(i)

# def step(s, a, actions, rng=None):
#     return grid_world.step(s, a, actions, rng)

# def oracle_model(s, action):
#     """Oracle model that predicts the next state for diagonal actions (4-7)"""
#     s_model_next, r_model, done_model = grid_world.step(grid_world.from_index(s), action, ACTIONS_8)
#     return s_model_next

class Visualizer:
    """Visualization utilities for GridWorld policies and Q-tables"""
    ARROWS = {0: '↑', 1: '↓', 2: '←', 3: '→', 4: '↘', 5: '↙', 6: '↗', 7: '↖'}

    @staticmethod
    def derive_policy(Q, n_rows, n_cols):
        """Derive policy from Q-table"""
        policy = np.zeros((n_rows, n_cols), dtype=int)
        for i in range(n_rows):
            for j in range(n_cols):
                state_index = i * n_cols + j
                policy[i, j] = np.argmax(Q[state_index])
        return policy

    @staticmethod
    def render_policy(policy, n_rows, n_cols, walls, start, goal):
        """Render policy with arrows"""
        print("\nPolicy Visualization:")
        for i in range(n_rows):
            row_str = ""
            for j in range(n_cols):
                if (i, j) == start:
                    action = policy[i, j]
                    arrow = Visualizer.ARROWS.get(action, '?')
                    row_str += " S "
                elif (i, j) == goal:
                    row_str += " G "
                elif (i, j) in walls:
                    row_str += " ▓ "
                elif (i, j) in CLIFF:
                    row_str += " C "
                else:
                    action = policy[i, j]
                    arrow = Visualizer.ARROWS.get(action, '?')
                    row_str += f" {arrow} "
            print(row_str)

    @staticmethod
    def print_value_grid(Q, n_rows, n_cols):
        """Print Q-table values as a grid"""
        print(f"\nQ-table shape: {Q.shape}")
        V = np.max(Q, axis=1).reshape(n_rows, n_cols)
        print(f"Value function (max Q-values):")
        for i in range(n_rows):
            row_str = ""
            for j in range(n_cols):
                row_str += f"{V[i, j]:6.1f} ?"
            print(row_str)

In [ ]:
EPISODES = 600

grid_world = GridWorld(
    n_rows=N_ROWS,
    n_cols=N_COLS,
    start=START,
    goal=GOAL,
    walls = WALLS,
    step_reward= -0.1,
    goal_reward=1,
    bump_reward=-0.2,
    gamma=0.95
)

def oracle_model(s, action):
    """Oracle model that predicts the next state for all 8 actions (including diagonal actions 4-7)"""
    s_model_next, r_model, done_model = grid_world.step(grid_world.from_index(s), action, ACTIONS_8)
    return s_model_next, r_model, done_model

In [ ]:
  # 1. Q-learning with 4 actions
agent_plain_4_action = QLearningAgent(grid_world, 4, seed=42)
agent_plain_4_action.train(ACTIONS_4, epsilon_greedy)
results_plain = agent_plain_4_action.get_results(moving_average)

# ----------------------------------------
# policy4 = Visualizer.derive_policy(agent_plain_4_action.Q, N_ROWS, N_COLS)
# Visualizer.render_policy(policy4, N_ROWS, N_COLS, WALLS, START, GOAL)
#Visualizer.print_value_grid(agent_plain_4_action.Q, N_ROWS, N_COLS)

# 3. Oracle Q-learning with 5 actions 
# 5.Q-learning (non-conditional) orange curve
oracle_agent = OracleQLearningAgent(grid_world = grid_world, n_actions = len(ACTIONS_5),
                                        episodes = EPISODES, base_q_table=copy.deepcopy(agent_plain_4_action.Q),
                                        use_conditional=True,
                                        use_model=True,
                                        seed=23)

oracle_agent.train_with_oracle(ACTIONS_5, epsilon_greedy, oracle_model)
policy5 = Visualizer.derive_policy(oracle_agent.Q, N_ROWS, N_COLS)
Visualizer.render_policy(policy5, N_ROWS, N_COLS, WALLS, START, GOAL)
# # # ----------------------------------------

In [ ]:
TEST_ACTION = ACTIONS_8
def run_multiple_experiments(n_runs=5, base_seed=123):
    """Run multiple experiments with different seeds and return statistics"""
    
    # Storage for all runs
    all_returns4 = []
    all_returns5plain = []
    all_returns5oracle = []
    all_returns5mlp = []
    all_returns5mlp_no_cond = []
    all_bumps4 = []
    all_bumps5plain = []
    all_bumps5oracle = []
    all_bumps5mlp = []
    all_bumps5mlp_no_cond = []
    all_bumps5ignore = []
    all_reuse_counts_mlp = []
    all_reuse_counts_oracle = []
    
    for run in range(n_runs):
        current_seed = base_seed + run * 42
        # Reset random seeds
        rng = np.random.default_rng(current_seed)
        random.seed(current_seed)
        np.random.seed(current_seed)
        torch.manual_seed(current_seed)
        
        print(f"Running experiment {run + 1}/{n_runs}...")
        
        # 1. Q-learning with 4 actions
        agent_plain_4_action = QLearningAgent(grid_world = grid_world, n_actions= len(ACTIONS_4), episodes = EPISODES, seed=current_seed)
        agent_plain_4_action.train(ACTIONS_4, epsilon_greedy)
        results_plain = agent_plain_4_action.get_results(moving_average)
        all_returns4.append(results_plain['returns'])
        all_bumps4.append(results_plain['bumps'])
        
        # 4. MLP Q-learning with learned model (conditional)
        mlp_agent = MLPQLearningAgentUseCondition(grid_world = grid_world,
                                      n_actions = len(TEST_ACTION),
                                      episodes = EPISODES,
                                      use_model=True, 
                                      use_conditional=True,
                                      base_q_table=copy.deepcopy(agent_plain_4_action.Q), seed=current_seed)
        mlp_agent.train_with_learned_model(TEST_ACTION, epsilon_greedy)
        mlp_results = mlp_agent.get_results(moving_average)
        all_returns5mlp.append(mlp_results['returns'])
        all_bumps5mlp.append(mlp_results['bumps'])
        all_reuse_counts_mlp.append(mlp_agent.reuse_count)
        # 5.Q-learning (non-conditional) orange curve
        mlp_agent_no_cond = OracleQLearningAgent(grid_world = grid_world, 
                                                n_actions = len(TEST_ACTION),
                                                episodes = EPISODES,
                                                base_q_table=copy.deepcopy(agent_plain_4_action.Q),
                                                use_conditional=False,
                                                use_model=True,
                                                seed=current_seed)
        
        mlp_agent_no_cond.train_with_oracle(TEST_ACTION, epsilon_greedy, oracle_model)
        mlp_results_no_cond = mlp_agent_no_cond.get_results(moving_average)
        all_returns5mlp_no_cond.append(mlp_results_no_cond['returns'])
        all_bumps5mlp_no_cond.append(mlp_results_no_cond['bumps'])
        # red curve
        oracle_agent_run = OracleQLearningAgent(grid_world = grid_world,
                                                n_actions = len(TEST_ACTION),
                                                episodes = EPISODES,
                                                base_q_table=copy.deepcopy(agent_plain_4_action.Q),
                                                use_conditional=True,
                                                use_model=True,
                                                seed=current_seed)
        
        oracle_agent_run.train_with_oracle(TEST_ACTION, epsilon_greedy, oracle_model)
        oracle_results_run = oracle_agent_run.get_results(moving_average)
        all_returns5oracle.append(oracle_results_run['returns'])
        all_bumps5oracle.append(oracle_results_run['bumps'])
        all_reuse_counts_oracle.append(oracle_agent_run.reuse_count)
        # black curve
        agent_plain = QLearningAgent(grid_world = grid_world,
                                     n_actions = len(TEST_ACTION),
                                     episodes = EPISODES, seed=current_seed)
        agent_plain.train(TEST_ACTION, epsilon_greedy)
        results_plain = agent_plain.get_results(moving_average)
        all_returns5plain.append(results_plain['returns'])
        all_bumps5plain.append(results_plain['bumps'])
        
    # Calculate statistics
    all_returns4 = np.array(all_returns4)
    all_returns5plain = np.array(all_returns5plain)
    all_returns5oracle = np.array(all_returns5oracle)
    all_returns5mlp = np.array(all_returns5mlp)
    all_returns5mlp_no_cond = np.array(all_returns5mlp_no_cond)
    all_bumps4 = np.array(all_bumps4)
    all_bumps5plain = np.array(all_bumps5plain)
    all_bumps5oracle = np.array(all_bumps5oracle)
    all_bumps5mlp = np.array(all_bumps5mlp)
    all_bumps5mlp_no_cond = np.array(all_bumps5mlp_no_cond)
    all_bumps5ignore = np.array(all_bumps5ignore)
    
    # Calculate moving averages for all runs
    ma_w = 25
    all_ret_ma4 = np.array([moving_average(returns, w=ma_w) for returns in all_returns4])
    all_ret_ma5plain = np.array([moving_average(returns, w=ma_w) for returns in all_returns5plain])
    all_ret_ma5oracle = np.array([moving_average(returns, w=ma_w) for returns in all_returns5oracle])
    all_ret_ma5mlp = np.array([moving_average(returns, w=ma_w) for returns in all_returns5mlp])
    all_ret_ma5mlp_no_cond = np.array([moving_average(returns, w=ma_w) for returns in all_returns5mlp_no_cond])
    
    all_bumps4_avg = np.array([moving_average(bumps, w=ma_w) for bumps in all_bumps4])
    all_bumps5plain_avg = np.array([moving_average(bumps, w=ma_w) for bumps in all_bumps5plain])
    all_bumps5oracle_avg = np.array([moving_average(bumps, w=ma_w) for bumps in all_bumps5oracle])
    all_bumps5mlp_avg = np.array([moving_average(bumps, w=ma_w) for bumps in all_bumps5mlp])
    all_bumps5mlp_no_cond_avg = np.array([moving_average(bumps, w=ma_w) for bumps in all_bumps5mlp_no_cond])
    
    return {
        'returns': {
            '4_actions': {'mean': np.mean(all_ret_ma4, axis=0), 'std': np.std(all_ret_ma4, axis=0)},
            '5_plain': {'mean': np.mean(all_ret_ma5plain, axis=0), 'std': np.std(all_ret_ma5plain, axis=0)},
            '5_oracle': {'mean': np.mean(all_ret_ma5oracle, axis=0), 'std': np.std(all_ret_ma5oracle, axis=0)},
            '5_mlp': {'mean': np.mean(all_ret_ma5mlp, axis=0), 'std': np.std(all_ret_ma5mlp, axis=0)},
            '5_mlp_no_cond': {'mean': np.mean(all_ret_ma5mlp_no_cond, axis=0), 'std': np.std(all_ret_ma5mlp_no_cond, axis=0)},
        },
        'bumps': {
            '4_actions': {'mean': np.mean(all_bumps4_avg, axis=0), 'std': np.std(all_bumps4_avg, axis=0)},
            '5_plain': {'mean': np.mean(all_bumps5plain_avg, axis=0), 'std': np.std(all_bumps5plain_avg, axis=0)},
            '5_oracle': {'mean': np.mean(all_bumps5oracle_avg, axis=0), 'std': np.std(all_bumps5oracle_avg, axis=0)},
            '5_mlp': {'mean': np.mean(all_bumps5mlp_avg, axis=0), 'std': np.std(all_bumps5mlp_avg, axis=0)},
            '5_mlp_no_cond': {'mean': np.mean(all_bumps5mlp_no_cond_avg, axis=0), 'std': np.std(all_bumps5mlp_no_cond_avg, axis=0)},
        },
        'info': {
            'mlp_reuse_counts': all_reuse_counts_mlp,
            'oracle_reuse_counts': all_reuse_counts_oracle
        }
    }

# Run multiple experiments
print("Running multiple experiments for statistical analysis...")
stats = run_multiple_experiments(n_runs=20)  # Reduced runs for faster execution

# Show the reuse_count captured from MLP agent per run
print('MLP reuse counts per run:', stats['info']['mlp_reuse_counts'])
print('Oracle reuse counts per run:', stats['info']['oracle_reuse_counts'])

# Plot results with shaded error bars
def plot_with_shaded_errors(stats, figsize=(12, 6)):
    """Plot results with shaded error bars"""
    
    # Returns plot
    plt.figure(figsize=figsize)
    
    # Calculate x-axis for each series (they might have different lengths due to moving average)
    episodes_total = EPISODES
    ma_w = 25
    
    # Plot each method with shaded error bars
    methods = [
        #('4_actions', 'Q learning (4 actions)', 'blue'),
        ('5_plain', 'Q learning (5 actions)', 'black'), 
        ('5_oracle', 'Oracle-based adaptation (model use-conditional)', 'red'),
        ('5_mlp_no_cond', 'Oracle-based adaptation (non-conditional)', 'orange'),
        ('5_mlp', 'MLP-based adaptation (learned model)', 'green'),
    ]
    
    for method_key, label, color in methods:
        mean_vals = stats['returns'][method_key]['mean']
        std_vals = stats['returns'][method_key]['std']
        x_vals = np.arange(episodes_total - len(mean_vals), episodes_total)
        
        # Plot mean line
        plt.plot(x_vals, mean_vals, color=color, label=f'Return: {label}', linewidth=2)
        
        # Plot shaded error region (mean ± std)
        plt.fill_between(x_vals, mean_vals - std_vals, mean_vals + std_vals, 
                        color=color, alpha=0.2)
    
    plt.xlabel('Episode')
    plt.ylabel('Discounted return')
    plt.title('Q-learning in GridWorld: Comparison of Adaptation Methods')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Bumps plot
    plt.figure(figsize=figsize)
    
    for method_key, label, color in methods:
        mean_vals = stats['bumps'][method_key]['mean']
        std_vals = stats['bumps'][method_key]['std']
        x_vals = np.arange(len(mean_vals))
        
        # Plot mean line
        plt.plot(x_vals, mean_vals, color=color, label=f'Total Bumps: {label}', linewidth=2)
        
        # Plot shaded error region (mean ± std)
        plt.fill_between(x_vals, mean_vals - std_vals, mean_vals + std_vals, 
                        color=color, alpha=0.2)
    
    plt.xlabel('Episode')
    plt.ylabel('Total Bumps')
    plt.title('Q-learning in GridWorld: Collision Comparison')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot the results with shaded error bars
plot_with_shaded_errors(stats)
